# Tutorial 6: Artefact-learning vs process-learning in fixed text environments

This notebook accompanies Tutorial 6 of the appendix. It demonstrates:
1. How two text environments are constructed from the same arithmetic problems
2. How a small causal transformer learns different behaviours from each
3. The environment × deployment mode interaction

**Dependencies:** This notebook uses the existing experiment infrastructure from
`theorem2_process_learning.py` and `theorem2_environment_modes.py`.
The experiments have already been run and results saved; this notebook loads
and visualises them. To re-run from scratch, set `RUN_EXPERIMENTS = True`.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Find project root
def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if p.name == 'Drift_and_selection':
            return p
        if (p / 'GitHub').exists() and (p / 'Nat_Paper').exists():
            return p
    return start

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / 'GitHub' / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_ROOT = PROJECT_ROOT / 'GitHub' / 'data' / 'outputs' / 'theorem2_process_learning'
FIG_ROOT = PROJECT_ROOT / 'GitHub' / 'appendix' / 'figures' / 'generated'
PROJECT_ROOT

## 1. What the text environments look like

Both environments use the same underlying addition problems.
The only difference is what tokens appear in the training target.

In [ ]:
# Show concrete examples of each environment
from drift_selection.theorem2_environment_modes import make_environment_example

a, b = 725, 639
for style in ['artifact_only', 'worked_trace', 'failed_then_repair']:
    ex = make_environment_example(a, b, style, prompt_mode='DIRECT', 
                                  split_name='demo', example_id='demo_001')
    print(f'\n=== {style.upper()} ===')
    print(f'Prompt:  {" ".join(ex["prompt_tokens"])}')
    print(f'Target:  {" ".join(ex["target_tokens"])}')
    print(f'Answer:  {" ".join(ex["answer_digits"])}')

## 2. Load existing experiment results

The experiments have been run with:
- 9,000 training / 1,200 val / 2,000 test (in-distribution) / 1,600 test (OOD)
- 2-layer causal transformer (d=128, 4 heads, d_ff=512, ~445K params)
- 8 epochs, batch 64, lr=3e-4, AdamW

In [ ]:
# Task A results
task_a_dir = DATA_ROOT / 'theorem2_taskA_addition_process_vs_artifact__sanity_in_dist_ep10'
task_a_summary = pd.read_csv(task_a_dir / 'evaluation' / 'summary_metrics.csv')
print('Task A: Process vs Artefact')
display(task_a_summary[['style', 'answer_accuracy']])

In [ ]:
# Task B results - direct mode
task_b_dir = DATA_ROOT / 'taskB_environment_modes_v2' / 'theorem2_taskB_environment_modes_v2'
task_b_direct = pd.read_csv(task_b_dir / 'evaluation' / 'summary_direct_mode.csv')
task_b_process = pd.read_csv(task_b_dir / 'evaluation' / 'summary_process_mode.csv')

print('Task B: Direct-answer mode (in-distribution)')
display(task_b_direct[task_b_direct['split'] == 'id_test'][
    ['style', 'direct_exact_match', 'direct_clean_answer_rate', 'direct_marker_contamination_rate']])

print('\nTask B: Show-work mode (in-distribution)')
display(task_b_process[task_b_process['split'] == 'id_test'][
    ['style', 'process_final_answer_accuracy', 'process_trace_validity']])

## 3. Training dynamics

The worked-trace environment converges much faster because the intermediate
tokens provide dense supervision at each step.

In [ ]:
# Load training histories
art_hist = pd.read_csv(task_b_dir / 'models' / 'artifact_only' / 'metrics_history.csv')
wt_hist = pd.read_csv(task_b_dir / 'models' / 'worked_trace' / 'metrics_history.csv')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for i, col in enumerate(['train_loss', 'val_loss']):
    ax = axes[i]
    ax.plot(art_hist['epoch'], art_hist[col], 'o-', color='#2196F3', label='Artefact-only')
    ax.plot(wt_hist['epoch'], wt_hist[col], 's--', color='#FF9800', label='Worked-trace')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Cross-entropy loss')
    ax.set_title(f'({"ab"[i]}) {"Training" if i==0 else "Validation"} loss')
    ax.legend()
    ax.set_ylim(bottom=0)
fig.tight_layout()
plt.show()

## 4. Sample generations

Concrete examples of what each trained model produces.

In [ ]:
# Show sample generations from each environment
for style in ['artifact_only', 'worked_trace']:
    for mode in ['direct', 'process']:
        csv_name = f'sample_generations_{style}_id_test_{mode}.csv'
        csv_path = task_b_dir / 'evaluation' / csv_name
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            print(f'\n=== {style} / {mode} mode (first 3 examples) ===')
            for _, row in df.head(3).iterrows():
                print(f'  {row["a"]} + {row["b"]} = {row["gold_answer"]}')
                tok = str(row.get('prediction_tokens', ''))
                if len(tok) > 120:
                    tok = tok[:120] + '...'
                print(f'  Model output: {tok}')
                print()

## 5. The environment × deployment heatmap

This is the key figure for the tutorial: neither environment dominates.

In [ ]:
# Build heatmap from results
id_direct = task_b_direct[task_b_direct['split'] == 'id_test'].set_index('style')
id_process = task_b_process[task_b_process['split'] == 'id_test'].set_index('style')

envs = ['artifact_only', 'worked_trace']
env_labels = ['Artefact-only', 'Worked-trace']
modes = ['Direct (clean)', 'Direct (exact)', 'Show-work (ans)', 'Show-work (trace)']

data = np.array([
    [id_direct.loc[e, 'direct_clean_answer_rate'],
     id_direct.loc[e, 'direct_exact_match'],
     id_process.loc[e, 'process_final_answer_accuracy'],
     id_process.loc[e, 'process_trace_validity']]
    for e in envs
])

fig, ax = plt.subplots(figsize=(8, 3.5))
im = ax.imshow(data, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(modes)))
ax.set_xticklabels(modes, fontsize=10)
ax.set_yticks(range(len(envs)))
ax.set_yticklabels(env_labels, fontsize=11)
for i in range(len(envs)):
    for j in range(len(modes)):
        color = 'white' if data[i,j] < 0.3 or data[i,j] > 0.7 else 'black'
        ax.text(j, i, f'{data[i,j]:.1%}', ha='center', va='center', fontsize=12, 
                fontweight='bold', color=color)
fig.colorbar(im, ax=ax, label='Accuracy', shrink=0.8)
ax.set_title('Environment × deployment mode interaction', fontsize=12)
fig.tight_layout()
plt.show()

## 6. Key takeaways

1. **Same problems, different text → different learner behaviour.** The artefact-only environment gives clean direct answers; the worked-trace environment gives valid derivations.

2. **Neither environment dominates.** Each excels in its matching deployment mode.

3. **This is about learner capacity, not population statistics.** An n-gram model cannot exploit worked traces (it lacks memory). The environment × deployment interaction requires a learner with sufficient internal capacity — here, a small transformer.

4. **Connection to Theorem 3:** What the learner inherits is the conditional distribution in its training environment. Different environments → different conditionals → different deployment behaviour.